# Debug IW residual DIS² against DIS²
Load a hybrid **CSV** (flat results include the equation components), optionally add historical DIS² CSV/pickle results, and inspect one dataset/shift/model at a time across representations. Nothing is trained or written by this notebook.

The current residual estimator uses **full source**:
\[
U_{IW}=R_O^{IW}+m_N e_S+m_N d_{T|N}-m_N d_S,\qquad L_{IW}=\operatorname{clip}(1-U_{IW},0,1).
\]
DIS² reports \(L=\operatorname{clip}(1-[e_S+\Delta+C],0,1)\). Its correction \(C\) is included; the hybrid is a **plug-in estimate without a confidence correction**. Historical DIS² uses different splits. Historical CSVs/pickles may only contain the combined discrepancy, so it is shown as one component.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/lib/hybrid_validation.py").exists())
sys.path.insert(0, str(ROOT))
from src.plot.compare_iw_hybrids import load_results, _NumpyCompatUnpickler

# The CLI loader imports Agg; restore notebook rendering AFTER that import.
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")

def read_historical(path):
    if Path(path).suffix == ".csv":
        return pd.read_csv(path)
    try:
        return load_results([path])
    except TypeError:
        # Handle historical pickles saved with newer pandas string dtypes.
        with open(path, "rb") as handle:
            return _NumpyCompatUnpickler(handle).load()

# EDIT: select exactly one new run, not a mixture of runs.
HYBRID_CSV = ROOT / "results/debug_comparison/iw_hybrid_REPLACE_WITH_RUN_ID.csv"
HISTORICAL = ROOT / "results/dis2_50epochs_30repeats_valfrac0.50.pkl"  # or None / a CSV

# None selects the first available case; set explicit values after viewing the inventory.
DATASET = None
SHIFT = None
TRAIN_METHOD = None
SEED = 0
IW_THRESHOLD = 2.0
REPRESENTATIONS = ["logits", "features", "PCA1", "PCA4"]
ACCURACY_COLUMN = "trg_accuracy"  # full target; or trg_eval_accuracy (historical may be unavailable)


In [ ]:
if not HYBRID_CSV.exists():
    raise FileNotFoundError(f"Set HYBRID_CSV to your result CSV. Found: {list((ROOT / 'results').rglob('iw_hybrid_*.csv'))[:15]}")
new = pd.read_csv(HYBRID_CSV)
new = new[new.prediction_method.isin(["iw_residual_dis2", "dis2_reference"])].copy()
assert not new.empty, "No residual hybrid/reference rows in this file."
required = ["dataset", "shift", "train_method", "bound_strategy", "lower_bound", "trg_accuracy"]
assert set(required).issubset(new), f"Missing columns: {set(required) - set(new)}"
if "schema_version" not in new or not (new.schema_version == 3).all():
    print("WARNING: these results do not all use schema 3 (current shared-training protocol).")
residual = new[new.prediction_method == "iw_residual_dis2"]
if "residual_source_region" not in residual or not (residual.residual_source_region == "full").all():
    print("WARNING: full-source residual semantics are unverified. Older runs used source non-overlap; rerun for the current equation.")
frames = [new]
if HISTORICAL is not None:
    old = read_historical(HISTORICAL)
    old = old.copy()
    old["prediction_method"] = "dis2_historical"
    frames.append(old)
data = pd.concat(frames, ignore_index=True)
for col in ["dataset", "shift", "train_method", "bound_strategy"]:
    data[col] = data[col].astype(str)
data["shift"] = data["shift"].str.replace(r"\.0$", "", regex=True)
new = data[data.prediction_method != "dis2_historical"]
display(new.groupby(["dataset", "shift", "train_method", "bound_strategy"]).size().rename("rows").reset_index())


## All available target accuracies and lower bounds
This overview uses **all datasets, shifts, models, seeds, thresholds and representations in the loaded CSV**, independent of the single-case filters. Each panel is one representation; each IW threshold has a separate series. Historical DIS² is restricted to matching cases. Points above the diagonal overestimate accuracy. Multiple seeds remain separate points; the counts are rows, not independent datasets. The hybrid has no confidence correction, while DIS² includes its saved correction.


In [ ]:
# Overview ignores the single-case filters above: all new-run cases, seeds and W values.
# Historical rows are restricted to cases/representations present in the new run.
case_keys = ["dataset", "shift", "train_method", "bound_strategy"]
overview_new = data[data.prediction_method.isin(["iw_residual_dis2", "dis2_reference"])].copy()
case_truth = overview_new[case_keys + ["trg_accuracy"]].drop_duplicates()
assert not case_truth.duplicated(case_keys).any(), "Inconsistent ground truth within new-run cases."
overview_old = data[data.prediction_method.eq("dis2_historical")].merge(
    case_truth, on=case_keys, how="inner", suffixes=("", "_new"), validate="many_to_one")
assert not overview_old.duplicated(case_keys).any(), "Duplicate historical cases."
assert np.allclose(overview_old.trg_accuracy, overview_old.trg_accuracy_new, atol=1e-6), "Historical ground truth differs from the new run."
overview = pd.concat([overview_new, overview_old.drop(columns=["trg_accuracy_new"])], ignore_index=True)
# Use full-target ground truth consistently: historical held-out accuracy may be unavailable.
valid = np.isfinite(overview.trg_accuracy) & np.isfinite(overview.lower_bound)
if "status" in overview:
    valid &= overview.status.isna() | overview.status.eq("ok")
print(f"Overview: {valid.sum()} plotted rows; {(~valid).sum()} unsupported/nonfinite rows omitted.")
overview = overview[valid].copy()
assert not overview.empty, "No finite estimates available for the overview."
representations = list(dict.fromkeys(overview_new.bound_strategy))
ncols = min(3, len(representations))
nrows = int(np.ceil(len(representations) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4.5*nrows),
                         squeeze=False, sharex=True, sharey=True)
overview["series"] = overview.prediction_method.replace({
    "dis2_reference": "DIS² reference", "dis2_historical": "DIS² historical"})
is_iw = overview.prediction_method.eq("iw_residual_dis2")
overview.loc[is_iw, "series"] = overview.loc[is_iw, "iw_threshold"].map(
    lambda w: f"IW residual DIS² (plug-in), W={w:g}")
series = list(dict.fromkeys(overview.series))
colors = dict(zip(series, plt.get_cmap("tab10").colors * (1 + len(series)//10)))
stats = []
for ax, rep in zip(axes.flat, representations):
    subset = overview[overview.bound_strategy.eq(rep)]
    for label in series:
        group = subset[subset.series.eq(label)]
        if group.empty:
            continue
        ax.scatter(group.trg_accuracy, group.lower_bound, s=24, alpha=.65,
                   color=colors[label], label=f"{label} (n={len(group)})")
        gap = group.trg_accuracy - group.lower_bound
        stats.append({"representation": rep, "method": label, "n": len(group),
                      "fraction_above_truth": float((gap < 0).mean()),
                      "mean_truth_minus_bound": float(gap.mean())})
    ax.plot([0, 1], [0, 1], "k--", linewidth=1)
    ax.set(title=rep, xlabel="Ground-truth target accuracy (full target)",
           ylabel="Saved accuracy lower bound / plug-in estimate", xlim=(0, 1), ylim=(0, 1))
    ax.grid(alpha=.2)
    if not subset.empty:
        ax.legend(fontsize=7, loc="best")
for ax in list(axes.flat)[len(representations):]:
    ax.set_visible(False)
fig.suptitle("All available cases: IW residual and DIS²", fontsize=14)
fig.tight_layout()
plt.show()
display(pd.DataFrame(stats))


In [ ]:
selected = new.copy()
for col, value in [("dataset", DATASET), ("shift", SHIFT), ("train_method", TRAIN_METHOD)]:
    value = str(value) if value is not None else selected.iloc[0][col]
    selected = selected[selected[col] == value]
    assert not selected.empty, f"No rows for {col}={value}"
case = selected.iloc[0]
mask = np.ones(len(data), dtype=bool)
for col in ["dataset", "shift", "train_method"]:
    mask &= data[col].eq(case[col])
view = data[mask & data.bound_strategy.isin(REPRESENTATIONS)].copy()
view = view[(view.prediction_method == "dis2_historical") | (view.seed == SEED)]
view = view[(view.prediction_method != "iw_residual_dis2") | np.isclose(view.iw_threshold, IW_THRESHOLD)]
assert not view.empty, "No matching seed, threshold or representations."
assert not view.duplicated(["prediction_method", "bound_strategy"]).any(), "Duplicate cases: select a single run/seed/threshold."
if "status" in view:
    bad = view.status.notna() & view.status.ne("ok")
    if bad.any():
        print("Unsupported rows (not plotted):")
        display(view.loc[bad, ["prediction_method", "bound_strategy", "status"]])
    view = view[~bad].copy()
assert not view.empty, "All matching rows are unsupported."
assert np.allclose(view.trg_accuracy, view.trg_accuracy.iloc[0], atol=1e-6), "Full-target ground truth differs across matched rows; check models/data."
view["label"] = view.prediction_method + " / " + view.bound_strategy
print(f"Case: {case.dataset} / shift {case['shift']} / {case.train_method}; seed {SEED}; W={IW_THRESHOLD}")
cols = ["prediction_method", "bound_strategy", "lower_bound", "accuracy_estimate_raw", "trg_accuracy", "trg_eval_accuracy", "epsilon", "n_target_b", "iw_effective_n", "schema_version"]
display(view[[c for c in cols if c in view]].reset_index(drop=True))


## Saved lower bounds and target accuracy
One point per method and representation. Missing combinations are left blank; full features are not silently substituted for historical PCA1. Ground truth uses the selected accuracy column; historical held-out accuracy may be absent.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
methods = ["iw_residual_dis2", "dis2_reference", "dis2_historical"]
for j, method in enumerate(methods):
    subset = view[view.prediction_method == method]
    x = [REPRESENTATIONS.index(v) + (j-1)*.16 for v in subset.bound_strategy]
    ax.scatter(x, subset.lower_bound, label=method, s=55)
    if ACCURACY_COLUMN in subset:
        ax.scatter(x, subset[ACCURACY_COLUMN], marker="x", color="black", s=45)
ax.scatter([], [], marker="x", color="black", label=f"Ground truth ({ACCURACY_COLUMN})")
ax.set(xticks=range(len(REPRESENTATIONS)), xticklabels=REPRESENTATIONS,
       ylabel="Accuracy", ylim=(-.03, 1.03), title="Saved lower bounds / hybrid plug-in estimate")
ax.grid(axis="y", alpha=.25)
ax.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## Signed contributions to the error expression
Hybrid bars are already multiplied by the target non-overlap mass. The IW contribution is already an unnormalized contribution: **do not multiply it by overlap mass again**. Positive and negative bars are stacked separately; their signed sum is the error expression before clipping. Diamonds show that sum. The table checks reconstruction against the saved values. DIS² discrepancy is shown jointly because separate source/target disagreements are not exported for the reference or historical runs.

In [ ]:
components = []
for _, row in view.iterrows():
    if row.prediction_method == "iw_residual_dis2":
        mass = row.target_residual_mass
        # A zero residual mass contributes zero even when conditional means are absent.
        term = lambda col: 0.0 if mass == 0 else mass * row.get(col, np.nan)
        terms = {"IW overlap": row.iw_error_contribution,
                 "m_N * source error": term("residual_source_error"),
                 "m_N * target disagreement": term("residual_target_disagreement"),
                 "-m_N * source disagreement": -term("residual_source_disagreement")}
    else:
        terms = {"source error": 1-row.h_val_acc,
                 "discrepancy": row.max_ts_agree_diff, "correction C": row.epsilon}
    assert np.isfinite(list(terms.values())).all(), f"Missing components for {row.label}; inspect the CSV."
    total = sum(terms.values())
    reconstructed = np.clip(1-total, 0, 1)
    assert np.isclose(reconstructed, row.lower_bound, atol=1e-6), f"Reconstruction mismatch for {row.label}"
    if row.prediction_method == "iw_residual_dis2":
        assert np.isclose(total, row.error_estimate_raw, atol=1e-6)
    components.append({"label": row.label, **terms, "error_sum": total,
                       "accuracy_before_clipping": 1-total, "saved_lower_bound": row.lower_bound})
summary = pd.DataFrame(components).set_index("label")
term_columns = [c for c in summary if c not in ["error_sum", "accuracy_before_clipping", "saved_lower_bound"]]
terms = summary[term_columns].fillna(0)
fig, ax = plt.subplots(figsize=(max(10, len(terms)*.8), 5))
pos = np.zeros(len(terms)); neg = pos.copy()
for col in terms:
    values = terms[col].to_numpy()
    ax.bar(range(len(terms)), values, bottom=np.where(values >= 0, pos, neg), label=col)
    pos += np.maximum(values, 0); neg += np.minimum(values, 0)
ax.scatter(range(len(terms)), summary.error_sum, marker="D", color="black", label="Signed total", zorder=5)
ax.axhline(0, color="black", linewidth=.8)
ax.set(xticks=range(len(terms)), xticklabels=terms.index, ylabel="Contribution to error", title="Error expression before accuracy conversion and clipping")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
ax.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()
display(summary.round(6))
